# Explainable Hybrid PQD Classification — Fixed & Final
**Three Contributions:** 1D Residual TCN · Noise Immunity Profile · Grad-CAM Explainability  
**Root cause fixed:** TCN now trains on **clean data** → high accuracy. Noise robustness is tested separately (not mixed into training, which caused val_loss explosion).


In [ ]:
import os, glob
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

tf.random.set_seed(42); np.random.seed(42)
print("TF:", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))


In [ ]:
# ── Load XPQRS dataset ────────────────────────────────────────────────────────
DATA_DIR = "/kaggle/input/seed-power-quality-disturbance-dataset/XPQRS"
csv_paths = glob.glob(os.path.join(DATA_DIR, "*.csv"))
assert csv_paths, f"No CSVs found in {DATA_DIR}"

dfs = []
for fp in csv_paths:
    label = os.path.splitext(os.path.basename(fp))[0]
    df0   = pd.read_csv(fp)
    dfm   = df0.melt(var_name="instance", value_name="amplitude")
    dfm["time_idx"] = dfm.groupby("instance").cumcount()
    dfm["label"]    = label
    dfs.append(dfm)

full_df = pd.concat(dfs, ignore_index=True)
pivot   = full_df.pivot_table(index=["label","instance"], columns="time_idx", values="amplitude")
X_raw   = pivot.values.astype("float32")
labels  = pivot.index.get_level_values("label")

le          = LabelEncoder()
y           = le.fit_transform(labels)
num_classes = len(le.classes_)
print("Samples:", X_raw.shape[0], "| Classes:", num_classes)
print(list(le.classes_))


In [ ]:
# ── Per-sample normalisation + train/test split ───────────────────────────────
def normalise(X):
    mu  = X.mean(axis=1, keepdims=True)
    sig = X.std(axis=1,  keepdims=True) + 1e-8
    return (X - mu) / sig

X_norm = normalise(X_raw)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_norm, y, test_size=0.2, stratify=y, random_state=42)

# (N, 999, 1)  — channel dim for Conv1D
X_train = X_train_raw[..., None].astype("float32")
X_test  = X_test_raw[..., None].astype("float32")
print("Train:", X_train.shape, "| Test:", X_test.shape)


## Phase 1 — Residual Denoising Autoencoder (DAE)
Trained across random SNR 20–50 dB so it generalises to real-world conditions.  
Used as an *optional pre-processor* during the noise immunity profile — **not** injected into TCN training.


In [ ]:
def add_awgn(X, snr_db):
    """Add Additive White Gaussian Noise at a given SNR (dB). X: (N, T, 1)."""
    sig_pow   = np.mean(X**2, axis=(1,2), keepdims=True)
    noise_pow = sig_pow / (10.0 ** (snr_db / 10.0))
    return X + np.random.normal(0, 1, X.shape).astype("float32") * np.sqrt(noise_pow)

# ── Build Residual DAE ────────────────────────────────────────────────────────
inp = layers.Input((999, 1))
x   = layers.Conv1D(32, 5, padding="same", activation="relu")(inp)
x   = layers.MaxPool1D(2, padding="same")(x)
x   = layers.Conv1D(16, 5, padding="same", activation="relu")(x)
x   = layers.MaxPool1D(2, padding="same")(x)
x   = layers.Conv1D(16, 5, padding="same", activation="relu")(x)
x   = layers.UpSampling1D(2)(x)
x   = layers.Conv1D(32, 5, padding="same", activation="relu")(x)
x   = layers.UpSampling1D(2)(x)
x   = layers.Conv1D(1,  5, padding="same")(x)
x   = layers.Cropping1D((0, 1))(x)
out = layers.Add()([inp, x])           # residual skip: predict correction, not clean signal
dae = Model(inp, out, name="Residual_DAE")
dae.compile(optimizer="adam", loss="mse")
dae.summary()


In [ ]:
# ── Train DAE with mixed SNR ──────────────────────────────────────────────────
rng = np.random.default_rng(42)
SNR_TRAIN = [20, 25, 30, 35, 40, 45, 50]

for ep in range(1, 16):
    losses = []
    idx = rng.permutation(len(X_train))
    for i in range(0, len(X_train), 32):
        b      = idx[i:i+32]
        snr    = rng.choice(SNR_TRAIN)
        noisy  = add_awgn(X_train[b], snr)
        losses.append(dae.train_on_batch(noisy, X_train[b]))
    val_noisy = add_awgn(X_test, 25)
    val_loss  = dae.evaluate(val_noisy, X_test, verbose=0)
    print(f"DAE Epoch {ep:02d}/15 | train={np.mean(losses):.4f} | val@25dB={val_loss:.4f}")


## Phase 2 — 1D Residual TCN (Contribution 1)

| Fix | Problem | Solution |
|-----|---------|----------|
| `padding="same"` | `"causal"` padded early timesteps with zeros, diluting GlobalAvgPool | `"same"` uses the full cycle symmetrically |
| `GlobalMaxPooling1D` | Average pooling diluted short transients (≤20 samples) to ~2% signal | Max pooling finds the strongest feature *anywhere* in the cycle |
| `Adam` (not AdamW) | Weight decay shrank weights too aggressively on the small 1360-sample dataset | Standard Adam converges reliably here |
| **Train on clean data** | Noise augmentation during training caused train/val distribution mismatch → val_loss explosion | Train on clean; test robustness *separately* in Phase 3 |


In [ ]:
def residual_tcn_block(x, filters, dilation_rate, dropout=0.2):
    # FIX: padding="same" — non-causal, symmetric receptive field
    h = layers.Conv1D(filters, 3, padding="same", dilation_rate=dilation_rate)(x)
    h = layers.BatchNormalization()(h)
    h = layers.Activation("relu")(h)
    h = layers.Dropout(dropout)(h)
    h = layers.Conv1D(filters, 3, padding="same", dilation_rate=dilation_rate)(h)
    h = layers.BatchNormalization()(h)
    if x.shape[-1] != filters:
        x = layers.Conv1D(filters, 1, padding="same")(x)
    return layers.Activation("relu")(layers.Add()([x, h]))

def build_tcn(input_shape=(999, 1), num_classes=17):
    inp = layers.Input(shape=input_shape)
    x   = inp
    for d in [1, 2, 4, 8, 16, 32, 64]:
        x = residual_tcn_block(x, 64, d, dropout=0.2)
    # Named layer for Grad-CAM
    x = layers.Conv1D(128, 3, padding="same", activation="relu",
                      name="target_conv_layer")(x)
    # FIX: GlobalMaxPooling — detects strongest feature, not diluted average
    x   = layers.GlobalMaxPooling1D()(x)
    x   = layers.Dropout(0.4)(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    return Model(inp, out, name="1D_Residual_TCN_Fixed")

tcn_model = build_tcn(input_shape=(999, 1), num_classes=num_classes)
# FIX: Standard Adam — no weight decay on small dataset
tcn_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
tcn_model.summary()


In [ ]:
# ── Train on CLEAN data ───────────────────────────────────────────────────────
# KEY FIX: No noise augmentation here. The TCN learns clean signal features.
# Noise robustness is evaluated separately in Phase 3 (not mixed into training).

train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
            .shuffle(2000, seed=42).batch(32).prefetch(tf.data.AUTOTUNE))
val_ds   = (tf.data.Dataset.from_tensor_slices((X_test, y_test))
            .batch(32).prefetch(tf.data.AUTOTUNE))

cbs = [
    tf.keras.callbacks.ReduceLROnPlateau("val_loss", factor=0.5,
                                          patience=5, min_lr=1e-5, verbose=1),
    tf.keras.callbacks.EarlyStopping("val_accuracy", patience=12,
                                      restore_best_weights=True, verbose=1),
]

history = tcn_model.fit(train_ds, validation_data=val_ds,
                         epochs=80, callbacks=cbs, verbose=1)

preds    = np.argmax(tcn_model.predict(X_test, verbose=0), axis=1)
clean_acc = accuracy_score(y_test, preds)
print(f"\n✓ Clean Test Accuracy: {clean_acc:.4f}")
print(classification_report(y_test, preds, target_names=le.classes_))

# Training curves
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
a1.plot(history.history["accuracy"], label="Train")
a1.plot(history.history["val_accuracy"], label="Val")
a1.set_title("TCN Accuracy"); a1.legend(); a1.grid(alpha=0.3)
a2.plot(history.history["loss"], label="Train")
a2.plot(history.history["val_loss"], label="Val")
a2.set_title("TCN Loss"); a2.legend(); a2.grid(alpha=0.3)
plt.suptitle("1D Residual TCN — Training History", fontweight="bold")
plt.tight_layout(); plt.savefig("tcn_training.png", dpi=150); plt.show()

# Confusion matrix
cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(13, 11))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f"Confusion Matrix — TCN (acc={clean_acc:.4f})", fontweight="bold")
plt.xticks(rotation=45, ha="right", fontsize=8); plt.yticks(fontsize=8)
plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=150); plt.show()


## Phase 3 — Noise Immunity Profile (Contribution 2)
Compare the TCN (alone) vs DAE → TCN across SNR levels 15–50 dB.  
The TCN was trained on clean data, so this is a *fair* out-of-distribution robustness test.


In [ ]:
SNR_LEVELS = [15, 20, 25, 30, 35, 40, 45, 50]
acc_tcn, acc_dae = [], []

print(f"{'SNR':>6}  {'TCN':>8}  {'DAE+TCN':>10}")
print("─" * 30)
for snr in SNR_LEVELS:
    X_noisy = add_awgn(X_test, snr)

    p1 = np.argmax(tcn_model.predict(X_noisy, verbose=0), axis=1)
    a1 = accuracy_score(y_test, p1)

    X_clean = dae.predict(X_noisy, verbose=0)
    p2 = np.argmax(tcn_model.predict(X_clean, verbose=0), axis=1)
    a2 = accuracy_score(y_test, p2)

    acc_tcn.append(a1); acc_dae.append(a2)
    print(f"{snr:>6}  {a1*100:>7.2f}%  {a2*100:>9.2f}%")

plt.figure(figsize=(10, 5))
plt.plot(SNR_LEVELS, [a*100 for a in acc_tcn],  "o--", color="#e74c3c",
         lw=2, ms=8, label="TCN only")
plt.plot(SNR_LEVELS, [a*100 for a in acc_dae],  "s-",  color="#2ecc71",
         lw=2.5, ms=8, label="DAE → TCN")
plt.axhline(clean_acc*100, color="steelblue", ls=":", lw=1.5,
            label=f"Clean baseline ({clean_acc*100:.1f}%)")
plt.fill_between(SNR_LEVELS, [a*100 for a in acc_tcn],
                              [a*100 for a in acc_dae],
                 alpha=0.15, color="#2ecc71", label="DAE gain")
plt.xlabel("SNR (dB)"); plt.ylabel("Accuracy (%)")
plt.title("Contribution 2: Noise Immunity Profile", fontweight="bold")
plt.xticks(SNR_LEVELS); plt.ylim([0, 105])
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("noise_immunity.png", dpi=150); plt.show()


## Phase 4 — Grad-CAM Explainability (Contribution 3)
Projects model attention onto the waveform — highlights *which milliseconds* triggered the classification.


In [ ]:
def gradcam_1d(signal_1d, model, layer_name, class_idx):
    """Returns normalised 1D heatmap (999,) for signal_1d (999,)."""
    grad_model = tf.keras.Model(
        model.inputs,
        [model.get_layer(layer_name).output, model.output]
    )
    x = tf.convert_to_tensor(signal_1d[None, :, None], dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x)
        conv_out, preds = grad_model([x], training=False)
        score = preds[:, class_idx]
    grads        = tape.gradient(score, conv_out)          # (1, T', C)
    pooled       = tf.reduce_mean(grads, axis=(0, 1))      # (C,)
    heatmap      = (conv_out[0].numpy() @ pooled.numpy())  # (T',)
    heatmap      = np.maximum(heatmap, 0)
    heatmap     /= heatmap.max() + 1e-8
    if len(heatmap) < 999:
        heatmap = np.interp(np.linspace(0, len(heatmap)-1, 999),
                             np.arange(len(heatmap)), heatmap)
    return heatmap, preds[0].numpy()

# ── Plot Grad-CAM for 6 representative classes ────────────────────────────────
show_classes = [c for c in ["Pure_Sinusoidal","Sag","Swell","Harmonics",
                              "Transient","Oscillatory_Transient"] if c in le.classes_]
if not show_classes:
    show_classes = list(le.classes_[:6])

fig, axes = plt.subplots(len(show_classes), 1, figsize=(16, len(show_classes)*3.2))
t_axis = np.linspace(0, 20, 999)

for ax, cls_name in zip(axes, show_classes):
    cls_idx    = le.transform([cls_name])[0]
    candidates = np.where((y_test == cls_idx) & (preds == cls_idx))[0]
    if len(candidates) == 0:
        candidates = np.where(y_test == cls_idx)[0]
    sig        = X_test[candidates[0]].squeeze()   # (999,)

    heat, prob = gradcam_1d(sig, tcn_model, "target_conv_layer", cls_idx)

    ax.plot(t_axis, sig, color="#2c3e50", lw=1.3, label="Waveform", zorder=3)
    ax.fill_between(t_axis, heat * sig.max() * 0.9, alpha=0.45,
                    color="#e74c3c", label="Grad-CAM", zorder=2)
    ax.set_title(f"{cls_name.replace('_',' ')}  (conf: {prob[cls_idx]*100:.1f}%)",
                 fontsize=11, fontweight="bold")
    ax.set_ylabel("Amplitude"); ax.legend(loc="upper right", fontsize=9)
    ax.grid(alpha=0.25)

axes[-1].set_xlabel("Time (ms)")
fig.suptitle("Contribution 3: Grad-CAM — Red = model attention region",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout(); plt.savefig("gradcam.png", dpi=150, bbox_inches="tight"); plt.show()
print("✓ Grad-CAM saved.")


In [ ]:
# ── Summary table ────────────────────────────────────────────────────────────
rows = [
    ("1D Residual TCN (clean test)",        f"{clean_acc*100:.2f}%", "Contribution 1"),
    ("TCN @ 30 dB SNR",  f"{acc_tcn[SNR_LEVELS.index(30)]*100:.2f}%", "Out-of-dist"),
    ("DAE+TCN @ 30 dB",  f"{acc_dae[SNR_LEVELS.index(30)]*100:.2f}%", "Contribution 2"),
    ("DAE+TCN @ 25 dB",  f"{acc_dae[SNR_LEVELS.index(25)]*100:.2f}%", "Contribution 2"),
    ("DAE+TCN @ 20 dB",  f"{acc_dae[SNR_LEVELS.index(20)]*100:.2f}%", "Contribution 2"),
    ("Grad-CAM",         "Visual",                                       "Contribution 3"),
]
df = pd.DataFrame(rows, columns=["Experiment","Accuracy","Contribution"])
print(df.to_string(index=False))


In [ ]:
# ── Export models ────────────────────────────────────────────────────────────
tcn_model.save("tcn_stage2.keras")
dae.save("dae.keras")
try:
    import tf2onnx
    spec = (tf.TensorSpec((None, 999, 1), tf.float32, name="input"),)
    tf2onnx.convert.from_keras(tcn_model, input_signature=spec,
                                opset=13, output_path="tcn_stage2.onnx")
    print("ONNX exported: tcn_stage2.onnx")
except ImportError:
    print("tf2onnx not installed — skipping ONNX export")

print("Done! Saved: tcn_stage2.keras | dae.keras")
